In [13]:
import os
import torch
import onnx
import onnxoptimizer

from lib.model.resNet.resNet import ResNet
from configs.test import parser

In [11]:
checkpoint_dir = "./output/checkpoints/"
onnxDir = "./output/onnx/"

checkSession = "1"
checkEpocj = "10"
checkPoint = "469"
device = "cpu"

In [12]:
resNet18 = ResNet()
resNet18.adaptMnist()
# load checkpoint
# path
checkpointName = 'resNet_{}_{}_{}.pth'.format(
    checkSession, checkEpocj, checkPoint)
print(">>> load checkpoint : {}".format(checkpointName))
checkpointPath = os.path.join(
    checkpoint_dir, str(checkSession), str(checkpointName))
resNet18.adaptMnist()
resNet18.loadCheckpoint(checkpointPath, device)

# onnx path
if not os.path.exists(onnxDir):
    os.makedirs(onnxDir)
onnxPath = os.path.join(onnxDir, checkpointName.replace('.pth', '.onnx'))
print(">>> export onnx to : {}".format(onnxPath))

>>> Initializing ResNet model.
>>> conv1 weight shape: (3, 64) - > (1, 64)
>>> fc weight shape: (512, 1000) - > (512, 10)
>>> load checkpoint : resNet_1_10_469.pth
>>> Initializing ResNet model.
>>> conv1 weight shape: (3, 64) - > (1, 64)
>>> fc weight shape: (512, 1000) - > (512, 10)
>>> checkpoint loaded
>>> export onnx to : ./output/onnx/resNet_1_10_469.onnx


d:\WorkSpace\NS_ResNet\lib\model\resNet\resNet.py:43: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpointPath, map_location=device)


In [15]:
bs = 1
inputSize = 256
inputChannel = 1
dummy = torch.randn(
    bs, inputChannel, inputSize, inputSize, 
    dtype=torch.float32, device=device)
torch.onnx.export(resNet18, dummy, onnxPath, verbose=False, opset_version=12, 
                  input_names=['input'], output_names=['output'], 
                  dynamic_axes=None)
modelOnnx = onnx.load(onnxPath)
optmodel = onnxoptimizer.optimize(modelOnnx, ["eliminate_identity"])
onnx.save(optmodel, onnxPath)